# EfficientNet-B3 + Multimodal (T2 + DWI) — fastMRI Prostate Cancer
**Input:** T2 + ADC + DWI TraceW + Calc B-Value (4 channels)  
**Architecture:** EfficientNet-B3 with modified 4-channel first conv layer  
**Author:** Akshaya Ganesh

In [1]:
# Cell 1 — Mount Drive and extract DICOMs
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import tarfile

DATA_DIR  = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS'
os.makedirs(DICOM_DIR, exist_ok=True)

print('Extracting DICOMs...')
with tarfile.open(f'{DATA_DIR}/fastmri_prostate_DICOMS_IDS_001_312.tar', 'r') as tar:
    tar.extractall(DICOM_DIR)

DICOM_DIR = '/content/DICOMS/DICOMS'
print(f'Patients: {len(os.listdir(DICOM_DIR))}')

Mounted at /content/drive
Extracting DICOMs...


/tmp/ipykernel_1860/2693115195.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DICOM_DIR)


Patients: 312


In [2]:
# Cell 2 — Installs
!pip install pydicom h5py optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 47.1 MB/s eta 0:00:00


In [3]:
# Cell 3 — Imports
import os
import numpy as np
import pandas as pd
import tarfile
import pydicom
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from PIL import Image
import torchvision.transforms as transforms
import optuna
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Cell 4 — Load labels and create splits
DATA_DIR  = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS/DICOMS'

labels_tar = f'{DATA_DIR}/labels.tar'
with tarfile.open(labels_tar, 'r') as tar:
    f = tar.extractfile('labels/t2_slice_level_labels.csv')
    t2_labels = pd.read_csv(f)
    f = tar.extractfile('labels/volume_exam_labels.csv')
    volume_labels = pd.read_csv(f)

volume_labels['binary_label'] = (volume_labels['t2_volume_level'] >= 3).astype(int)
patient_split = t2_labels.drop_duplicates('fastmri_pt_id')[['fastmri_pt_id', 'data_split']]
df = patient_split.merge(
    volume_labels[['fastmri_pt_id', 'binary_label', 't2_volume_level']],
    on='fastmri_pt_id'
)

invalid_ids = [115, 258]
train_df = df[df['data_split'] == 'training'].reset_index(drop=True)
val_df   = df[df['data_split'] == 'validation'].reset_index(drop=True)
test_df  = df[df['data_split'] == 'test'].reset_index(drop=True)

train_df = train_df[~train_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
val_df   = val_df[~val_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
test_df  = test_df[~test_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(df['binary_label'].value_counts())

Train: 217, Val: 47, Test: 46
binary_label
0    178
1    134
Name: count, dtype: int64


In [5]:
# Cell 5 — Check which sequences are available for a sample patient
sample_pt = os.path.join(DICOM_DIR, '001')
print(f'Sequences available for patient 001:')
for folder in sorted(os.listdir(sample_pt)):
    if not folder.startswith('.'):
        n_files = len([f for f in os.listdir(os.path.join(sample_pt, folder)) if f.endswith('.dcm')])
        print(f'  {folder}: {n_files} slices')

Sequences available for patient 001:
  AX_DIFFUSION_ADC: 30 slices
  AX_DIFFUSION_CALC_BVAL: 30 slices
  AX_DIFFUSION_TRACEW: 60 slices
  AX_T2: 30 slices


In [6]:
# Cell 6 — Check sequence availability across all patients
SEQUENCES = ['AX_T2', 'AX_DIFFUSION_ADC', 'AX_DIFFUSION_TRACEW', 'AX_DIFFUSION_CALC_BVAL']

missing = {seq: [] for seq in SEQUENCES}
for pt_id in df['fastmri_pt_id']:
    pt_path = os.path.join(DICOM_DIR, str(pt_id).zfill(3))
    for seq in SEQUENCES:
        seq_path = os.path.join(pt_path, seq)
        if not os.path.exists(seq_path) or len(os.listdir(seq_path)) == 0:
            missing[seq].append(pt_id)

print('Missing sequences per type:')
for seq, pts in missing.items():
    print(f'  {seq}: {len(pts)} patients missing')
    if pts:
        print(f'    IDs: {pts[:10]}')

# Find patients that have ALL sequences
all_missing = set()
for pts in missing.values():
    all_missing.update(pts)
print(f'\nPatients missing any sequence: {len(all_missing)}')
print(f'Patients with all sequences: {len(df) - len(all_missing)}')

Missing sequences per type:
  AX_T2: 0 patients missing
  AX_DIFFUSION_ADC: 0 patients missing
  AX_DIFFUSION_TRACEW: 1 patients missing
    IDs: [180]
  AX_DIFFUSION_CALC_BVAL: 0 patients missing

Patients missing any sequence: 1
Patients with all sequences: 311


In [7]:
# Cell 7 — Filter splits to patients with all sequences
# Add invalid_ids to missing set
all_missing.update(invalid_ids)

train_df = train_df[~train_df['fastmri_pt_id'].isin(all_missing)].reset_index(drop=True)
val_df   = val_df[~val_df['fastmri_pt_id'].isin(all_missing)].reset_index(drop=True)
test_df  = test_df[~test_df['fastmri_pt_id'].isin(all_missing)].reset_index(drop=True)

print(f'After filtering — Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(f'Train labels: {train_df.binary_label.value_counts().to_dict()}')

After filtering — Train: 217, Val: 47, Test: 45
Train labels: {0: 129, 1: 88}


In [13]:
# Cell 8 — Multimodal Dataset class (T2 + ADC + TRACEW + CALC_BVAL)
SEQUENCES = ['AX_T2', 'AX_DIFFUSION_ADC', 'AX_DIFFUSION_TRACEW', 'AX_DIFFUSION_CALC_BVAL']

def load_sequence_grid(pt_path, seq_name, target_size=(160, 160)):
    seq_path = os.path.join(pt_path, seq_name)
    slices   = []
    for fname in os.listdir(seq_path):
        if fname.endswith('.dcm'):
            ds  = pydicom.dcmread(os.path.join(seq_path, fname))
            arr = ds.pixel_array.astype(np.float32)
            # Resize to standard size to handle variable shapes
            img = Image.fromarray(arr)
            img = img.resize(target_size, Image.BILINEAR)
            arr = np.array(img)
            slices.append((int(ds.InstanceNumber), arr))
    slices.sort(key=lambda x: x[0])
    volume = np.stack([s[1] for s in slices])  # (N, H, W)

    # Per-volume min-max normalization
    volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

    # Take middle 9 slices
    mid      = volume.shape[0] // 2
    selected = volume[mid-4:mid+5]

    # Build 3x3 grid
    rows = []
    for i in range(3):
        row_img = np.concatenate([selected[i*3], selected[i*3+1], selected[i*3+2]], axis=1)
        rows.append(row_img)
    grid = np.concatenate(rows, axis=0)
    return grid


class ProstateMultimodalDataset(Dataset):
    def __init__(self, df, dicom_dir, augment=False):
        self.df        = df.reset_index(drop=True)
        self.dicom_dir = dicom_dir
        self.augment   = augment

        # Spatial transforms (applied identically to all channels)
        if augment:
            self.spatial_transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
            ])
        else:
            self.spatial_transform = transforms.Compose([
                transforms.Resize((224, 224)),
            ])

        # ImageNet normalization applied per channel using mean/std of each
        self.to_tensor = transforms.ToTensor()

        print(f'Multimodal Dataset: {len(self.df)} patients')
        print(f'Labels: {self.df.binary_label.value_counts().to_dict()}')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        pt_id  = str(row.fastmri_pt_id).zfill(3)
        pt_path = os.path.join(self.dicom_dir, pt_id)

        # Load each sequence as a 3x3 grid
        channel_tensors = []
        for seq in SEQUENCES:
            grid = load_sequence_grid(pt_path, seq)  # (H*3, W*3) float32 in [0,1]
            # Convert to PIL, apply spatial transforms, convert to tensor
            img  = Image.fromarray((grid * 255).astype(np.uint8))
            img  = self.spatial_transform(img)
            img  = self.to_tensor(img)  # (1, 224, 224)
            channel_tensors.append(img)

        # Stack all 4 sequences → (4, 224, 224)
        combined = torch.cat(channel_tensors, dim=0)

        # Normalize each channel independently using ImageNet stats
        # For 4th channel (no ImageNet stats) use mean of RGB stats
        mean = torch.tensor([0.485, 0.456, 0.406, 0.456]).view(4, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225, 0.226]).view(4, 1, 1)
        combined = (combined - mean) / std

        label = torch.tensor(row.binary_label, dtype=torch.float32)
        return combined, label

In [14]:
# Cell 9 — DataLoaders
train_dataset = ProstateMultimodalDataset(train_df, DICOM_DIR, augment=True)
val_dataset   = ProstateMultimodalDataset(val_df,   DICOM_DIR, augment=False)
test_dataset  = ProstateMultimodalDataset(test_df,  DICOM_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=0)

# Test one sample
sample_img, sample_label = train_dataset[0]
print(f'Image shape: {sample_img.shape}')  # should be (4, 224, 224)
print(f'Label: {sample_label}')
print(f'Min: {sample_img.min():.4f}, Max: {sample_img.max():.4f}')

Multimodal Dataset: 217 patients
Labels: {0: 129, 1: 88}
Multimodal Dataset: 47 patients
Labels: {1: 26, 0: 21}
Multimodal Dataset: 45 patients
Labels: {0: 26, 1: 19}
Image shape: torch.Size([4, 224, 224])
Label: 0.0
Min: -2.1179, Max: 1.3755


In [15]:
# Cell 10 — EfficientNet-B3 with 4-channel input and device
def get_model(dropout=0.5):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)

    # Modify first conv layer to accept 4 channels instead of 3
    original_conv = model.features[0][0]  # original 3-channel conv
    new_conv = nn.Conv2d(
        in_channels=4,
        out_channels=original_conv.out_channels,
        kernel_size=original_conv.kernel_size,
        stride=original_conv.stride,
        padding=original_conv.padding,
        bias=original_conv.bias is not None
    )
    # Initialize: copy RGB weights, initialize 4th channel as mean of RGB
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = original_conv.weight
        new_conv.weight[:, 3:, :, :] = original_conv.weight.mean(dim=1, keepdim=True)
    model.features[0][0] = new_conv

    # Freeze first 4 stages
    for i, block in enumerate(model.features):
        if i < 4:
            for param in block.parameters():
                param.requires_grad = False
    # Unfreeze the new first conv (it needs to learn the 4th channel)
    for param in model.features[0].parameters():
        param.requires_grad = True

    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 1)
    )
    return model


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Test forward pass with 4-channel input
test_model  = get_model().to(device)
test_input  = torch.randn(2, 4, 224, 224).to(device)
test_output = test_model(test_input)
print(f'Output shape: {test_output.shape}')  # should be (2, 1)
total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'Total params:     {total_params/1e6:.1f}M')
print(f'Trainable params: {trainable_params/1e6:.1f}M')
del test_model, test_input, test_output

Using device: cuda
Output shape: torch.Size([2, 1])
Total params:     10.7M
Trainable params: 10.5M


In [16]:
# Cell 11 — Train/Val functions
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze(1)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc

In [18]:
# Cell 12 — Optuna hyperparameter search
n_neg = (train_df['binary_label'] == 0).sum()
n_pos = (train_df['binary_label'] == 1).sum()

def objective(trial):
    lr_backbone  = trial.suggest_float('lr_backbone',  1e-7, 1e-4, log=True)
    lr_head      = trial.suggest_float('lr_head',      1e-5, 1e-3, log=True)
    dropout      = trial.suggest_float('dropout',      0.1,  0.6)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 0.1,  log=True)

    trial_model = get_model(dropout=dropout).to(device)

    backbone_params   = [p for p in trial_model.parameters()
                         if p.requires_grad and id(p) not in
                         set(id(p) for p in trial_model.classifier.parameters())]
    classifier_params = list(trial_model.classifier.parameters())

    optimizer = torch.optim.AdamW([
        {'params': backbone_params,   'lr': lr_backbone},
        {'params': classifier_params, 'lr': lr_head}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
    pw        = torch.tensor([n_neg / n_pos]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    best_val_auc     = 0
    patience         = 3
    patience_counter = 0

    for epoch in range(10):
        train_epoch(trial_model, train_loader, optimizer, criterion, device)
        scheduler.step()
        _, val_auc = val_epoch(trial_model, val_loader, criterion, device)

        if val_auc > best_val_auc:
            best_val_auc     = val_auc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_auc


study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
)
study.optimize(objective, n_trials=10, timeout=7200)

print('\nBest Multimodal EfficientNet trial:')
print(f'  Val AUC: {study.best_value:.4f}')
print(f'  Params: {study.best_params}')

[I 2026-05-18 14:49:10,236] A new study created in memory with name: no-name-669df7d2-4207-43a4-ac5d-10d0433f06aa
[I 2026-05-18 14:57:19,094] Trial 0 finished with value: 0.39743589743589747 and parameters: {'lr_backbone': 1.7498342238165143e-06, 'lr_head': 1.2425460584293783e-05, 'dropout': 0.3609812357201836, 'weight_decay': 0.09047912978319751}. Best is trial 0 with value: 0.39743589743589747.
[I 2026-05-18 15:05:18,539] Trial 1 finished with value: 0.6465201465201466 and parameters: {'lr_backbone': 8.917153223363091e-06, 'lr_head': 4.058793575804495e-05, 'dropout': 0.3750395241481761, 'weight_decay': 0.013640509681346702}. Best is trial 1 with value: 0.6465201465201466.
[I 2026-05-18 15:13:08,333] Trial 2 finished with value: 0.4340659340659341 and parameters: {'lr_backbone': 3.195189589795884e-07, 'lr_head': 5.9318075455467765e-05, 'dropout': 0.269096058516645, 'weight_decay': 0.0027054219795496857}. Best is trial 1 with value: 0.6465201465201466.
[I 2026-05-18 15:25:16,624] Trial


Best Multimodal EfficientNet trial:
  Val AUC: 0.6740
  Params: {'lr_backbone': 6.780169510204319e-06, 'lr_head': 5.408210796543673e-05, 'dropout': 0.2679052368355854, 'weight_decay': 0.0010589881869386272}


In [19]:
# Cell 13 — Initialize final model with best Optuna params
best = study.best_params
print(f'Training final model with: {best}')

model = get_model(dropout=best['dropout']).to(device)

backbone_params   = [p for p in model.parameters()
                     if p.requires_grad and id(p) not in
                     set(id(p) for p in model.classifier.parameters())]
classifier_params = list(model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {'params': backbone_params,   'lr': best['lr_backbone']},
    {'params': classifier_params, 'lr': best['lr_head']}
], weight_decay=best['weight_decay'])

scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
pos_weight = torch.tensor([n_neg / n_pos]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print('Ready')

Training final model with: {'lr_backbone': 6.780169510204319e-06, 'lr_head': 5.408210796543673e-05, 'dropout': 0.2679052368355854, 'weight_decay': 0.0010589881869386272}
Ready


In [20]:
# Cell 14 — Training loop
EPOCHS   = 50
PATIENCE = 10
best_val_auc     = 0
patience_counter = 0
best_model_path  = f'{DATA_DIR}/best_efficientnet_multimodal.pth'

for epoch in range(EPOCHS):
    train_loss, train_auc = train_epoch(model, train_loader, optimizer, criterion, device)
    scheduler.step()
    val_loss, val_auc     = val_epoch(model, val_loader, criterion, device)

    if val_auc > best_val_auc:
        best_val_auc     = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'Epoch {epoch+1:02d}/{EPOCHS} — Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} *** BEST ***')
    else:
        patience_counter += 1
        print(f'Epoch {epoch+1:02d}/{EPOCHS} — Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} (patience {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}.')
            break

print(f'\nBest Val AUC: {best_val_auc:.4f}')

Epoch 01/50 — Train Loss: 0.8366 AUC: 0.4253 | Val Loss: 0.9217 AUC: 0.5861 *** BEST ***
Epoch 02/50 — Train Loss: 0.8402 AUC: 0.4258 | Val Loss: 0.9081 AUC: 0.5128 (patience 1/10)
Epoch 03/50 — Train Loss: 0.8219 AUC: 0.5226 | Val Loss: 0.9100 AUC: 0.4560 (patience 2/10)
Epoch 04/50 — Train Loss: 0.8223 AUC: 0.5191 | Val Loss: 0.9232 AUC: 0.3333 (patience 3/10)
Epoch 05/50 — Train Loss: 0.8159 AUC: 0.5788 | Val Loss: 0.9268 AUC: 0.3022 (patience 4/10)
Epoch 06/50 — Train Loss: 0.8232 AUC: 0.5376 | Val Loss: 0.9202 AUC: 0.3168 (patience 5/10)
Epoch 07/50 — Train Loss: 0.8387 AUC: 0.4256 | Val Loss: 0.9178 AUC: 0.3205 (patience 6/10)
Epoch 08/50 — Train Loss: 0.8266 AUC: 0.5077 | Val Loss: 0.9188 AUC: 0.3352 (patience 7/10)
Epoch 09/50 — Train Loss: 0.8202 AUC: 0.5399 | Val Loss: 0.9160 AUC: 0.3535 (patience 8/10)
Epoch 10/50 — Train Loss: 0.8263 AUC: 0.4939 | Val Loss: 0.9098 AUC: 0.3700 (patience 9/10)
Epoch 11/50 — Train Loss: 0.8350 AUC: 0.4368 | Val Loss: 0.9067 AUC: 0.3810 (patien

In [21]:
# Cell 15 — Test evaluation
model.load_state_dict(torch.load(best_model_path))
_, test_auc = val_epoch(model, test_loader, criterion, device)
print(f'Multimodal EfficientNet Test AUC: {test_auc:.4f}')

Multimodal EfficientNet Test AUC: 0.6113


In [22]:
# Cell 16 — Save results
import json

multimodal_results = {
    'model': 'EfficientNet-B3 Multimodal',
    'approach': '3x3 grid, 4 channels: T2 + ADC + TraceW + CalcBval',
    'sequences': SEQUENCES,
    'best_val_auc': float(best_val_auc),
    'test_auc': float(test_auc),
    'best_params': study.best_params,
    'n_train': len(train_df),
    'n_val': len(val_df),
    'n_test': len(test_df),
    'epochs_trained': epoch + 1,
}

results_path = f'{DATA_DIR}/efficientnet_multimodal_results.json'
with open(results_path, 'w') as f:
    json.dump(multimodal_results, f, indent=2)

print('Results saved:')
print(json.dumps(multimodal_results, indent=2))

# Compare with T2-only EfficientNet
t2_only_path = f'{DATA_DIR}/efficientnet_results.json'
if os.path.exists(t2_only_path):
    with open(t2_only_path) as f:
        t2_results = json.load(f)
    print('\n=== Comparison ===')
    print(f'{"Model":<35} {"Val AUC":>10} {"Test AUC":>10}')
    print('-' * 55)
    print(f'{"EfficientNet-B3 (T2 only)":<35} {t2_results["best_val_auc"]:>10.4f} {t2_results["test_auc"]:>10.4f}')
    print(f'{"EfficientNet-B3 (Multimodal)":<35} {best_val_auc:>10.4f} {test_auc:>10.4f}')

Results saved:
{
  "model": "EfficientNet-B3 Multimodal",
  "approach": "3x3 grid, 4 channels: T2 + ADC + TraceW + CalcBval",
  "sequences": [
    "AX_T2",
    "AX_DIFFUSION_ADC",
    "AX_DIFFUSION_TRACEW",
    "AX_DIFFUSION_CALC_BVAL"
  ],
  "best_val_auc": 0.5860805860805861,
  "test_auc": 0.611336032388664,
  "best_params": {
    "lr_backbone": 6.780169510204319e-06,
    "lr_head": 5.408210796543673e-05,
    "dropout": 0.2679052368355854,
    "weight_decay": 0.0010589881869386272
  },
  "n_train": 217,
  "n_val": 47,
  "n_test": 45,
  "epochs_trained": 11
}

=== Comparison ===
Model                                  Val AUC   Test AUC
-------------------------------------------------------
EfficientNet-B3 (T2 only)               0.6813     0.5500
EfficientNet-B3 (Multimodal)            0.5861     0.6113
